In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(r"D:\IIT\Year 4\IRP\IRP_Project_Folder")

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "interim"
    / "weekly_demand_500_articles.csv"
)

weekly_dataset = pd.read_csv(
    DATA_PATH,
    dtype={"article_id": "string"},
    parse_dates=["week_start"]
)

print("Dataset loaded successfully!")
print("Total records:", len(weekly_dataset))
print("Unique articles:", weekly_dataset["article_id"].nunique())
print("Date range:", weekly_dataset["week_start"].min(),
      "to", weekly_dataset["week_start"].max())

print("\nColumn data types:")
print(weekly_dataset.dtypes)

weekly_dataset.head()

Dataset loaded successfully!
Total records: 20308
Unique articles: 500
Date range: 2018-09-17 00:00:00 to 2020-09-21 00:00:00

Column data types:
article_id                    string
week_start            datetime64[us]
weekly_demand                  int64
product_type_name                str
product_group_name               str
colour_group_name                str
department_name                  str
index_name                       str
section_name                     str
garment_group_name               str
dtype: object


,article_id,week_start,weekly_demand,product_type_name,product_group_name,colour_group_name,department_name,index_name,section_name,garment_group_name
0,0189616032,2019-12-09,20,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
1,0189616032,2019-12-16,45,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
2,0189616032,2019-12-23,38,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
3,0189616032,2019-12-30,29,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
4,0189616032,2020-01-06,32,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic


In [2]:
print("Missing values:")
print(weekly_dataset.isnull().sum())

print("\nDuplicate article-week combinations:")
print(
    weekly_dataset.duplicated(
        subset=["article_id", "week_start"]
    ).sum()
)

print("\nDemand value checks:")
print("Minimum demand:", weekly_dataset["weekly_demand"].min())
print("Maximum demand:", weekly_dataset["weekly_demand"].max())
print("Zero-demand records:",
      (weekly_dataset["weekly_demand"] == 0).sum())
print("Negative-demand records:",
      (weekly_dataset["weekly_demand"] < 0).sum())

Missing values:
article_id            0
week_start            0
weekly_demand         0
product_type_name     0
product_group_name    0
colour_group_name     0
department_name       0
index_name            0
section_name          0
garment_group_name    0
dtype: int64

Duplicate article-week combinations:
0

Demand value checks:
Minimum demand: 0
Maximum demand: 1096
Zero-demand records: 6262
Negative-demand records: 0


In [3]:
all_weeks = pd.date_range(
    start=weekly_dataset["week_start"].min(),
    end=weekly_dataset["week_start"].max(),
    freq="7D"
)

print("Total calendar weeks:", len(all_weeks))
print("First week:", all_weeks.min())
print("Last week:", all_weeks.max())

print("\nAre all dates Mondays?")
print((weekly_dataset["week_start"].dt.dayofweek == 0).all())

Total calendar weeks: 106
First week: 2018-09-17 00:00:00
Last week: 2020-09-21 00:00:00

Are all dates Mondays?
True


C:\Users\HP\AppData\Local\Temp\ipykernel_27628\3264269172.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  print((weekly_dataset["week_start"].dt.dayofweek == 0).all())


In [4]:
total_weeks = len(all_weeks)

train_count = int(np.ceil(total_weeks * 0.70))
validation_count = int(np.ceil(total_weeks * 0.15))
test_count = total_weeks - train_count - validation_count

train_weeks = all_weeks[:train_count]
validation_weeks = all_weeks[
    train_count:train_count + validation_count
]
test_weeks = all_weeks[
    train_count + validation_count:
]

print("Total weeks:", total_weeks)

print("\nTRAINING PERIOD")
print("Number of weeks:", len(train_weeks))
print("Start:", train_weeks.min())
print("End:", train_weeks.max())

print("\nVALIDATION PERIOD")
print("Number of weeks:", len(validation_weeks))
print("Start:", validation_weeks.min())
print("End:", validation_weeks.max())

print("\nTESTING PERIOD")
print("Number of weeks:", len(test_weeks))
print("Start:", test_weeks.min())
print("End:", test_weeks.max())

print("\nTotal allocated weeks:",
      len(train_weeks) + len(validation_weeks) + len(test_weeks))

Total weeks: 106

TRAINING PERIOD
Number of weeks: 75
Start: 2018-09-17 00:00:00
End: 2020-02-17 00:00:00

VALIDATION PERIOD
Number of weeks: 16
Start: 2020-02-24 00:00:00
End: 2020-06-08 00:00:00

TESTING PERIOD
Number of weeks: 15
Start: 2020-06-15 00:00:00
End: 2020-09-21 00:00:00

Total allocated weeks: 106


In [5]:
train_end = train_weeks.max()
validation_end = validation_weeks.max()

weekly_dataset["split"] = np.select(
    [
        weekly_dataset["week_start"] <= train_end,
        weekly_dataset["week_start"] <= validation_end
    ],
    [
        "train",
        "validation"
    ],
    default="test"
)

print(weekly_dataset["split"].value_counts())

print("\nUnique articles in each split:")
print(
    weekly_dataset.groupby("split")["article_id"].nunique()
)

print("\nRecords in each split:")
print(
    weekly_dataset.groupby("split").size()
)

split
train         14585
validation     3672
test           2051
Name: count, dtype: int64

Unique articles in each split:
split
test          198
train         500
validation    255
Name: article_id, dtype: int64

Records in each split:
split
test           2051
train         14585
validation     3672
dtype: int64


In [6]:
article_coverage = (
    weekly_dataset
    .pivot_table(
        index="article_id",
        columns="split",
        values="weekly_demand",
        aggfunc="count",
        fill_value=0
    )
    .reindex(columns=["train", "validation", "test"], fill_value=0)
)

print("Articles with records in all three periods:",
      ((article_coverage > 0).all(axis=1)).sum())

print("Articles with no training records:",
      (article_coverage["train"] == 0).sum())

print("Articles with no validation records:",
      (article_coverage["validation"] == 0).sum())

print("Articles with no testing records:",
      (article_coverage["test"] == 0).sum())

article_coverage.head(10)

Articles with records in all three periods: 198
Articles with no training records: 0
Articles with no validation records: 245
Articles with no testing records: 302


split,train,validation,test
article_id,,,
0189616032,11,16,15
0240840032,1,0,0
0266875015,6,0,0
0318914016,1,0,0
0327144031,65,0,0
0335896020,1,0,0
0351933052,59,16,2
0387348017,1,0,0
0387843037,1,0,0
